# Total cholesterol at or above the screening cut — Random Forest · PNS 2013

**FAPESP–Illinois undiagnosed NCD project** — Isabela Venancio da Silva (USP São Paulo) ·
Xuan Lin · Amogh Mannava (UIUC).

One model family, one outcome. Twelve notebooks share this shape, so any two of
them can be compared line for line.

| | |
|---|---|
| **Outcome** | `Z031` >= 200 mg/dL |
| **Threshold authority** | Conventional screening cut. SBC uses risk-stratified LDL targets rather than one diagnostic value |
| **Cohort** | adults who answer *no* to `Q060` (1 yes, 2 no), i.e. never told by a doctor they had this condition |
| **Expected build** | 5,944 rows, prevalence 32.0% |
| **Model** | Random Forest |
| **Dropped as condition-specific** | `dx_cholesterol` |

**Why this family.** Non-linear baseline. No scaling and no spline: a tree splits on age directly, so the spline would only add collinear columns.

## What this notebook does not decide

Preprocessing is frozen. Every variable decision comes from
`PNS_preprocessing_registry_v1.xlsx` through `pns_preprocess.py`: 117 declared
variables in, 91 out, 28 dropped, 2 composites, 5 renamed. **To change a
variable, change the workbook, not this notebook.**

Layer 1 (`build_matrix`) runs once and is deterministic. Layer 2
(`build_preprocessor`) — imputation, splines, encoding, scaling — is fitted
inside each cross-validation fold and never sees the test rows.

## Open with the group

PNS 2013 has no lipid-lowering medication item: `Q06204` records a recommendation, not use. This model therefore cannot define 'treated' the way the other two can, and the lipid definition itself is still open in the decision log: TC, LDL, HDL or any abnormal lipid give between 360 and 2,331 positives.

`n_medications` and `n_chronic` are sums over their blocks and are rebuilt by
`build_matrix` **after** the condition-specific drop. This is the trap that once
produced an AUC of 0.86 with sensitivity 1.000; do not compute either counter
anywhere in this notebook.

## Three findings from testing the shared build, for Amogh

Reproducing the 15/09/2026 numbers locally turned up three things in the files as
they stand on the Drive. None is worked around silently; all three are in
`pns_modelkit.py`, documented at the point of use.

1. **The diagnosis gates are still imputed.** `dx_diabetes` and `dx_cholesterol`
   carry `na_rule = implied:0` in the registry, so an unanswered gate is filled
   with zero and the row is declared undiagnosed. That is bug 1 of the build
   note, and it builds 7,851 and 7,244 rows instead of 6,832 and 5,944. Setting
   both cells to `mar`, as `dx_hypertension` already is, reproduces the reported
   cohorts exactly. Section 2 stops with this message if it happens.
2. **One-hot columns and their declared categories disagree on type.**
   `_roles()` declares the categories as strings and leaves the columns numeric,
   so `OneHotEncoder` refuses to fit at all. Cast back to string before the
   encoder.
3. **Three ordinal orders are declared descending.** `passive_smoke`,
   `diet_salt_perception` and `smoking_status` were reverse-coded in the values
   by `_recode_fixes`, but the registry still records the pre-reversal order, and
   `OrdinalEncoder` rejects unsorted numeric categories. Sorted ascending, which
   is what 'higher means more exposure' means after that recode.

---
# 1 · Setup

Installs, the data, and the run switches. Nothing here touches the science,
except section 1.3, where the research question is chosen.

## 1.1 · Install

**What this does.** Installs what Colab does not ship for this family: `optuna`,
plus `openpyxl` for the workbook.

**What to look for.** Nothing, unless it errors.

In [ ]:
!pip install -q optuna openpyxl

## 1.2 · Data and shared code

**What this does.** Clones the public repository, which holds the survey file,
both dictionaries, the frozen preprocessing (`pns_preprocess.py` and the
registry workbook) and the shared model helper (`pns_modelkit.py`).

**Drive is off by default**, since mounting asks for permission every session.
The run then writes to the session disk and zips itself at the end. Set
`MOUNT_DRIVE = True` to write into the shared folder instead, which also keeps
the built matrix between sessions and saves the minute it takes to rebuild.
Leave the `DRIVE = ...` line in place either way: the configuration cell reads
it whether or not anything is mounted.

**What to look for.** The two sha256 stamps printed by the build in section 2
must match across notebooks. They are what proves twelve runs used one matrix.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/isasaade-23/pns2013-lab-exams-en.git"
REPO     = "/content/pns2013-lab-exams-en"

# Clone, or update a clone this session already has: a session that started
# before the last commit would otherwise keep running the old pns_modelkit.
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"])
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, os.path.join(REPO, "pipeline"))

print("pipeline at", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                    capture_output=True, text=True).stdout.strip())

DATA     = os.path.join(REPO, "data", "pns2013_lab_exams.xlsx")
REGISTRY = os.path.join(REPO, "pipeline", "PNS_preprocessing_registry_v1.xlsx")

# Drive is optional and off by default, because mounting asks for permission
# every session. Set MOUNT_DRIVE = True to write straight into the shared
# folder and to keep the built matrix between sessions; left False, the run
# writes to the session disk and zips itself at the end.
#
# DRIVE is defined either way. Do not comment this line out: the configuration
# cell below reads it.
MOUNT_DRIVE = False
DRIVE = "/content/drive/MyDrive/FAPESP_Illinois"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", e)

print("data    ", os.path.exists(DATA))
print("registry", os.path.exists(REGISTRY))

## 1.3 · Configuration

**What this does.** Replaces what used to be command-line flags. Everything
downstream reads these.

| Switch | Meaning |
|---|---|
| `THRESHOLD` | `None` uses the guideline cut. The prespecified sensitivity analysis is `THRESHOLD = (190,)`; the LDL variant needs `Z033` and a registry change |
| `COHORT` | `"undiagnosed"` is the primary framing (Model A). `"all"` is diagnostic only |
| `N_TRIALS` | tuning budget. 30 finishes inside a free Colab session; `FULL = True` raises it to 100 |
| `RS` | 42, fixed, so the split is identical across the twelve notebooks |

**Runtime.** about 25 minutes at 30 trials, 80 at 100 — the slowest of the four.

In [ ]:
import importlib
import numpy as np, pandas as pd
import pns_preprocess as pp
import pns_modelkit as mk

# re-import, in case an older copy was imported earlier in this session
importlib.reload(pp); importlib.reload(mk)

OUTCOME   = "cholesterol"
MODEL     = "rf"
THRESHOLD = None          # None = guideline default; see the table above
COHORT    = "undiagnosed"
FULL     = False          # True -> 100 trials, the paper run
N_TRIALS = 100 if FULL else 30

# outputs: the shared folder when Drive is mounted, the session disk otherwise
BASE   = f"{DRIVE}/02_analysis" if os.path.isdir(DRIVE) else "/content/work"
OUTDIR = os.path.join(BASE, "outputs", "models")
BUILD  = os.path.join(BASE, "outputs", "matrices")
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(BUILD, exist_ok=True)

print("writing to", OUTDIR)

---
# 2 · The frozen matrix

**What this does.** Calls `build_matrix()` from `pns_preprocess.py` — or reuses
a cached build whose data and registry hashes still match — then asserts the
counts reported to the group: **5,944 rows at 32.0%**.

**What to look for.** If the assertion fails, stop. It means the registry or the
survey file moved, and no result from this notebook is comparable to the others
until that is understood.

In [ ]:
bundle = mk.load_or_build(OUTCOME, data=DATA, registry=REGISTRY,
                          outdir=BUILD, threshold=THRESHOLD, cohort=COHORT)
mk.check_frozen(bundle)

X, y = bundle["X"], bundle["y"]
print(f"\n{X.shape[0]} rows x {X.shape[1]} predictors, prevalence {y.mean():.1%}")
print("threshold authority:", bundle["threshold_authority"])
print("data sha", bundle["data_sha"], "| registry sha", bundle["registry_sha"])

a = mk.attrition(bundle, BUILD)
display(a) if a is not None else None

**Participant flow and roles.** The attrition table above is the row accounting
for the flow diagram. Below, where each column enters Layer 2.

In [ ]:
for k, v in bundle["roles"].items():
    print(f"{k:<10} {len(v):>3}  {', '.join(v[:6])}{' ...' if len(v) > 6 else ''}")

blocks = pd.Series({c: bundle["spec"][c]["block"]
                    for c in X.columns if c in bundle["spec"]}).value_counts()
print("\npredictors per block\n", blocks.to_string())

---
# 3 · Split and preprocessor

**What this does.** Stratified 80/20 at `random_state=42`, then builds the Layer 2
`ColumnTransformer`. For Random Forest: scaling **off**,
age spline **off**.

**What to look for.** Train and test prevalence should match to a decimal. The
preprocessor is passed *into* the pipeline, never fitted here.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

Xtr, Xte, ytr, yte = mk.split(bundle)
SPW = mk.pos_weight(ytr)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=mk.RS)

def preproc():
    return mk.preprocessor(bundle, spline=False, scale=False)

n_encoded = mk.preprocessor(bundle, spline=False, scale=False,
                            verbose=True).fit_transform(Xtr).shape[1]
print(f"{X.shape[1]} raw predictors -> {n_encoded} encoded columns")
print(f"positives: train {int(ytr.sum())}, test {int(yte.sum())} "
      f"(scale_pos_weight {SPW:.1f})")

---
# 4 · Fit

Two fits, reported side by side: library defaults, and an Optuna TPE search over
5-fold AUC. The comparison is the honest way to say whether tuning bought
anything — in the hypertension run it was worth about +0.006 AUC.

## 4.1 · Defaults

**What to look for.** The CV AUC here is the floor. For hypertension it should
land near 0.730; the tuned run near 0.736.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def make_est(params):
    return RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                  random_state=mk.RS, **params)

pipe_def = Pipeline([("prep", preproc()),
                     ("clf", RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                random_state=mk.RS))])

cv_def = cross_val_score(pipe_def, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=1)
print(f"default 5-fold CV AUC {cv_def.mean():.3f} +/- {cv_def.std():.3f}")

pipe_def.fit(Xtr, ytr)
row_def, p_def = mk.evaluate(pipe_def, Xte, yte, "Random Forest (default)")
print(f"test AUC {row_def['AUC_test']:.3f} "
      f"[{row_def['AUC_lo']:.3f}, {row_def['AUC_hi']:.3f}]")

## 4.2 · Tuned

**The slow cell.** About 25 minutes at 30 trials, 80 at 100 — the slowest of the four. The objective reports fold by
fold so the pruner can stop a hopeless trial early. The search space is the one
used in the 07/08/2026 revision, unchanged.

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

_DEPTH = {'none': None, '6': 6, '10': 10, '16': 16, '24': 24}
_FEATS = {'sqrt': 'sqrt', 'log2': 'log2', '0.3': 0.3, '0.5': 0.5}

def suggest(t):
    return {'n_estimators': t.suggest_int('n_estimators', 250, 600),
            'max_depth': _DEPTH[t.suggest_categorical('max_depth', list(_DEPTH))],
            'min_samples_leaf': t.suggest_int('min_samples_leaf', 1, 25),
            'max_features': _FEATS[t.suggest_categorical('max_features', list(_FEATS))]}

def objective(trial):
    params, scores = suggest(trial), []
    for k, (itr, iva) in enumerate(cv.split(Xtr, ytr)):
        pipe = Pipeline([("prep", preproc()), ("clf", make_est(params))])
        pipe.fit(Xtr.iloc[itr], ytr.iloc[itr])
        scores.append(roc_auc_score(ytr.iloc[iva],
                                    pipe.predict_proba(Xtr.iloc[iva])[:, 1]))
        trial.report(float(np.mean(scores)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=mk.RS),
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print(f"\nbest CV AUC {study.best_value:.3f} after {len(study.trials)} trials")
print(best)

pipe_tuned = Pipeline([("prep", preproc()),
                       ("clf", make_est(suggest(optuna.trial.FixedTrial(best))))])
pipe_tuned.fit(Xtr, ytr)
row_tuned, p_tuned = mk.evaluate(pipe_tuned, Xte, yte, "Random Forest (tuned)")
row_def["CV_AUC"], row_tuned["CV_AUC"] = cv_def.mean(), study.best_value
BEST_MODEL, BEST_P, BEST_PARAMS = pipe_tuned, p_tuned, best

---
# 5 · Results

Measured on the test half, which nothing above was fitted on.

## 5.1 · Metrics

**What to look for.** `AUC_lo`/`AUC_hi` is a 1,000-draw stratified bootstrap
interval on the test AUC. Two operating points are reported: Youden's J, and the
screening point that holds sensitivity at about 90% — the one a screening
instrument would actually be set at, and the one where PPV shows what the
prevalence costs.

In [ ]:
results = mk.metrics_frame([row_def, row_tuned])
display(results)

## 5.2 · Permutation importance

**What this does.** Shuffles one predictor at a time in the test set and measures
how far AUC falls, then sums the drops per registry block. Ten repeats over every predictor.

**What to look for.** The block table is the null-result table of the decision
log: for hypertension, access to care contributed exactly 0.000 and sleep less
than that. A block that suddenly matters for a new outcome is a finding; a block
that matters *too much* is usually leakage, and the first thing to check is
whether a counter was rebuilt.

In [ ]:
imp, blocks_imp = mk.permutation_report(BEST_MODEL, Xte, yte, bundle, n_repeats=10)
display(imp.head(20))
display(blocks_imp)

## 5.3 · Figures

ROC for both configurations, calibration of the better one, and the top 20
predictors. Written at 300 dpi.

In [ ]:
fig_roc = mk.plot_roc({row_def["model"]: (yte, p_def),
                       row_tuned["model"]: (yte, p_tuned)},
                      f"{OUTCOME} · {row_tuned['model']}")
fig_cal = mk.plot_calibration(yte, BEST_P, f"{OUTCOME} · calibration")
fig_imp = mk.plot_importance(imp, f"{OUTCOME} · permutation importance")

---
# 6 · Export

One folder per run, under `outputs/models/<outcome>/<family>/`: the metrics row,
the importance tables, the best parameters, the figures and a plain-text report
carrying the build hashes. The metrics files from the twelve runs concatenate
into the comparison table without further bookkeeping.

**Where it goes.** `PUSH_RESULTS = True` commits the folder to
[isasaade-23/pns2013-model-runs](https://github.com/isasaade-23/pns2013-model-runs),
which is private — these are unpublished results. It needs a GitHub token in
the Colab saved keys named `GITHUB_TOKEN`: a fine-grained token with
**Contents: read and write** on that repository, enabled for this notebook.
Without a token, or with `PUSH_RESULTS = False`, the run zips itself and
downloads instead.

In [ ]:
PUSH_RESULTS = True        # False -> zip and download instead

folder = mk.export(OUTDIR, OUTCOME, MODEL, bundle, results,
                   importance=imp, blocks=blocks_imp, best_params=BEST_PARAMS,
                   figures=[("roc", fig_roc), ("calibration", fig_cal),
                            ("importance", fig_imp)])

pushed = None
if PUSH_RESULTS:
    try:
        pushed = mk.push_results(folder)
    except Exception as e:
        print("not pushed:", e)

if pushed is None:
    z = mk.zip_folder(folder, f"/content/{OUTCOME}_{MODEL}.zip")
    try:
        from google.colab import files
        files.download(z)
    except Exception as e:
        print(e)

## 6.1 · Re-push a run that is already on disk

**What this does.** Sends the folder the cell above wrote, without refitting
anything. Use it when the export did not push: a session that cloned before the
last commit and ran an older `pns_modelkit`, a missing token, a connection that
dropped. Set `RETRY_PUSH = True` and run this cell alone.

**What to look for.** The link it prints. If it says no token, add `GITHUB_TOKEN`
to the Colab saved keys and run it again — the results are on disk either way,
and nothing has to be recomputed.

In [ ]:
RETRY_PUSH = False

if RETRY_PUSH:
    import importlib
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"])
    import pns_modelkit as mk
    importlib.reload(mk)
    mk.push_results(os.path.join(OUTDIR, OUTCOME, MODEL))